# Домашнее задание 7. Сборка конвейера CI/CD
Если у вас еще нет аккаунта в GitLab, вам нужно будет его создать:
1. Перейдите на [GitLab](https://gitlab.com/) и войдите в свой аккаунт.
2. Нажмите на кнопку New Project (Новый проект).
3. Выберите Create blank project (Создать пустой проект).
4. Укажите имя проекта и описание (по желанию).
5. Выберите уровень видимости проекта (Public).
6. Нажмите Create project (Создать проект).
7. Дополните файл .gitlab-ci.yml необходимыми джобами и отправьте в репозиторий.

**Репозиторий с реализацией:** https://github.com/TheodorWeiss/ml-model-deployment-hw7

**Docker-образ в GitHub Container Registry:** `ghcr.io/theodorweiss/ml-model-deployment-hw7:latest`

В репозитории лежат: Flask-приложение `app.py` с `/health` и `/predict`, `Dockerfile`, `docker-compose.canary.yml` с nginx-балансировщиком, четыре nginx-конфигурации для разных фаз canary, два GitHub Actions workflow (CI и Deploy), а также бонусный стек мониторинга Prometheus + Grafana с дашбордом, показывающим распределение трафика по версиям в реальном времени.

## 1. Настроить CI/CD-пайплайн для ML-сервиса с использованием GitLab




Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

Вам дан рабочий код пайплайна и черновик файла .gitlab-ci.yml. Перепишите yaml в [ячейке](#scrollTo=s55MrS66JXWs)


*Ожидаемый артефакт: список коммитов в [ячейке](#scrollTo=gErasBmRSHjb) и ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=F0uQqbe3iHqE)*    

In [3]:
%%sh
git config --global user.email "fedor.vays@gmail.com"
git config --global user.name "Федор Вайс"
git init
pip install scikit-learn numpy pandas -qqq
pip freeze > requirements.txt

Initialized empty Git repository in /content/.git/


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>


In [4]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Writing ml_pipeline.py


### Проверяем работоспособность пайплайна

In [5]:
!python ml_pipeline.py

Точность аccuracy: 1.00


In [6]:
%%writefile .gitlab-ci.yml
stages:
  - test

run_ml_pipeline:
  stage: test
  image: python:3.11-slim
  script:
    - echo "🎉 The job was automatically triggered by a $CI_PIPELINE_SOURCE event."
    - echo "🐧 This job is now running on a GitLab Runner."
    - echo "🔎 The name of your branch is $CI_COMMIT_REF_NAME and your repository is $CI_PROJECT_PATH."
    - echo "💡 The repository has been cloned to the runner."
    - echo "🖥️ The workflow is now ready to test your code on the runner."
    - echo "📁 Listing files in the repository:"
    - ls -la
    - echo "🚀 Running ML pipeline..."
    - pip install -r requirements.txt
    - python ml_pipeline.py
    - echo "🍏 This job finished successfully."

Writing .gitlab-ci.yml


In [7]:
!git add .gitlab-ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitLab"
!git log --oneline --decorate -5

[master (root-commit) a17346a] build(ml_pipeline.py) добавлен пайплайн GitLab
 2 files changed, 31 insertions(+)
 create mode 100644 .gitlab-ci.yml
 create mode 100644 ml_pipeline.py
a17346a (HEAD -> master) build(ml_pipeline.py) добавлен пайплайн GitLab


### Проверка статуса пайплайна

После настройки файла `.gitlab-ci.yml`, вы можете закоммитить изменения и запушить их в репозиторий.

GitLab автоматически запустит пайплайн, и вы сможете наблюдать за его выполнением в разделе CI/CD своего проекта.

Что нужно сделать:

1. Перейдите в свой проект на GitLab.
2. Нажмите на вкладку CI/CD и выберите Pipelines.
3. Вы увидите список запущенных пайплайнов. Нажмите на последний, чтобы увидеть выполнение.
4. Убедитесь, что все джобы выполнены успешно (отмечены зеленым цветом).
5. Приложите ссылку на статус выполнения в разделе Pipelines **своего** репозитория на GitLab.

Ссылка на выполненный GitLab pipeline:

https://gitlab.com/mipt7245319/ci-cd-pipeline-assembly/-/jobs/14297693791

## 2. Обосновать стратегию деплоя (развертывания, Blue-Green, Canary, Rolling, Shadow) и оценить влияние на риски




Изучите [инструмент](https://github.com/npryce/adr-tools) для учета архитектурных решений и запишите **причины**, по которым мы начали использовать стратегию деплоя и **риски**, к которым нас привело такое решение.



*Ожидаемый артефакт: архитектурное решение в формате ADR в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

# ADR-001: Использовать Canary Deployment для ML-сервиса

Date: 2026-05-10

## Status

Accepted

## Context

В проекте разворачивается ML-сервис с двумя версиями модели: `v1.0.0` и `v1.1.0`.

В коде отсутствует полноценная обработка ошибок, поэтому переводить весь трафик сразу на новую версию рискованно. Новая версия может запуститься успешно, но при этом давать некорректные ответы, увеличивать число ошибок или ухудшать качество предсказаний.

Нужно выбрать стратегию развертывания, которая позволит проверить новую версию постепенно, сохранить доступность стабильной версии и выполнить rollback при ошибках.

## Decision

We will use **Canary Deployment**.

Трафик будет распределяться через балансировщик Nginx между двумя версиями сервиса:

- `v1.0.0` - стабильная версия;
- `v1.1.0` - новая версия.

На первом этапе новая версия получит только небольшую часть трафика, например `10%`, а основная часть трафика останется на стабильной версии. Если проверка `/health` и `/predict` проходит успешно и ошибок не появляется, долю новой версии можно постепенно увеличить до `50%`, а затем до `100%`.

Если новая версия работает некорректно, выполняется rollback: весь трафик возвращается на `v1.0.0`.

## Considered Options

**Blue-Green Deployment** - хорошая альтернатива, потому что позволяет держать две версии сервиса и быстро переключаться между ними. Главный плюс простой rollback. Главный риск при полном переключении на новую версию ошибка может сразу затронуть весь трафик.

**Canary Deployment** - выбранный вариант. Он позволяет проверять новую версию постепенно и ограничивать риск небольшой долей пользователей. Это особенно важно для ML-модели, потому что сервис может быть технически доступен, но качество ответов новой модели может оказаться хуже.

**Rolling Deployment** - подходит для Kubernetes и постепенной замены pod'ов, но для текущего учебного проекта на Docker Compose такая стратегия сложнее и менее наглядна.

**Shadow Deployment** - самый безопасный для пользователей, потому что новая версия получает копию трафика, но её ответы не возвращаются пользователям. Однако эта стратегия сложнее в реализации и не показывает реальное распределение пользовательского трафика между версиями.

## Consequences

Положительные последствия:

- новая версия не получает сразу весь трафик;
- стабильная версия остаётся доступной;
- можно постепенно увеличивать долю новой версии;
- rollback выполняется через изменение конфигурации Nginx;
- стратегия подходит для проверки ML-модели с минимизацией риска.

Отрицательные последствия:

- часть пользователей всё равно может попасть на нестабильную версию;
- нужен балансировщик трафика;
- для наблюдения за canary в реальном времени в проект добавлен стек Prometheus + Grafana с дашбордом, показывающим распределение запросов и latency по версиям (см. репозиторий, папки prometheus/ и grafana/).

## 3. Реализовать стратегию развертывания

Реализуйте стратегию, выбранную на предыдущем [шаге](#scrollTo=hoQdM6SrJXXE).



*Ожидаемый артефакт: yaml в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

In [8]:
%%writefile docker-compose.canary.yml
services:
  ml_service_v1:
    image: ml-service:v1.0.0
    container_name: ml_service_v1
    environment:
      - MODEL_VERSION=v1.0.0
    restart: unless-stopped

  ml_service_v2:
    # Тот же образ - разница только в переменной окружения MODEL_VERSION
    image: ml-service:v1.0.0
    container_name: ml_service_v2
    environment:
      - MODEL_VERSION=v1.1.0
    restart: unless-stopped

  nginx:
    image: nginx:1.27-alpine
    container_name: ml_nginx
    ports:
      - "8090:80"
    volumes:
      # Имя файла берётся из переменной окружения NGINX_CONFIG.
      # По умолчанию (canary не активен) - 90/10
      - ./nginx/${NGINX_CONFIG:-nginx.90-10.conf}:/etc/nginx/conf.d/default.conf:ro
    depends_on:
      - ml_service_v1
      - ml_service_v2
    restart: unless-stopped

Overwriting docker-compose.canary.yml


In [ ]:
%%writefile nginx/nginx.90-10.conf
upstream ml_backend {
    server ml_service_v1:8000 weight=90;
    server ml_service_v2:8000 weight=10;
}

server {
    listen 80;

    location / {
        proxy_pass http://ml_backend;
    }
}

In [ ]:
%%writefile nginx/nginx.50-50.conf
upstream ml_backend {
    server ml_service_v1:8000 weight=50;
    server ml_service_v2:8000 weight=50;
}

server {
    listen 80;

    location / {
        proxy_pass http://ml_backend;
    }
}

In [ ]:
%%writefile nginx/nginx.100.conf
upstream ml_backend {
    server ml_service_v2:8000;
}

server {
    listen 80;

    location / {
        proxy_pass http://ml_backend;
    }
}

In [ ]:
%%writefile nginx/nginx.rollback.conf
upstream ml_backend {
    server ml_service_v1:8000;
}

server {
    listen 80;

    location / {
        proxy_pass http://ml_backend;
    }
}

Выбрана стратегия Canary. Реализуется через nginx с весами в upstream-блоке. Файлы конфигов лежат в папке `nginx/` репозитория.

#### Переключение фаз canary

Переключение делается **пересозданием nginx-контейнера** с другим bind-mount'ом - это надёжнее, чем редактирование файла «на лету» (на Windows Docker Desktop последнее работает нестабильно из-за прослойки WSL2 ↔ NTFS).

Старт стека (по умолчанию 90/10):

```bash
docker compose -f docker-compose.canary.yml -f docker-compose.monitoring.yml up -d
```

Переход на 50/50:

```bash
set NGINX_CONFIG=nginx.50-50.conf
docker compose -f docker-compose.canary.yml -f docker-compose.monitoring.yml up -d
```

Полный rollout на новую версию:

```bash
set NGINX_CONFIG=nginx.100.conf
docker compose -f docker-compose.canary.yml -f docker-compose.monitoring.yml up -d
```

Rollback при ошибках:

```bash
set NGINX_CONFIG=nginx.rollback.conf
docker compose -f docker-compose.canary.yml -f docker-compose.monitoring.yml up -d
```

Проверка трафика через единую точку входа (nginx):

```bash
curl http://localhost:8090/health
curl -X POST http://localhost:8090/predict -H "Content-Type: application/json" -d '{"x":[5.1,3.5,1.4,0.2]}'
```

Дополнительно реализован скрипт `scripts/check_canary.py`, который шлёт N запросов и считает распределение ответов между версиями. Прогон показал ожидаемое распределение для 90/10, 50/50 и 100% сценариев. Скриншоты переключений в реальном времени - в Grafana (см. бонусный мониторинг).

## 4. Спланировать A/B-тестирование для ML-модели

Вспомните материалы [семинара](https://colab.research.google.com/drive/1TM1yieSFhUqVxBferzbcexpAtK00lGYe?usp=sharing) и опишите параметры эксперимента.



*Ожидаемый артефакт: код в [ячейке](#scrollTo=OluzjqEhaIpM)*

In [ ]:
import numpy as np
from scipy.stats import fisher_exact, ttest_ind

# -----------------------------
# План A/B-теста для ML-модели
# -----------------------------

experiment = {
    "model_A": "v1.0.0 - текущая стабильная модель",
    "model_B": "v1.1.0 - новая модель",
    "traffic_split": "50% / 50%",
    "primary_metric": "accuracy",
    "technical_metric": "latency",
    "alpha": 0.05,
    "sample_size_per_group": 300,
    "hypothesis_H0": "Новая модель v1.1.0 не отличается от v1.0.0 по качеству.",
    "hypothesis_H1": "Новая модель v1.1.0 показывает статистически значимое отличие по качеству."
}

print("A/B test parameters:")
for key, value in experiment.items():
    print(f"- {key}: {value}")

# -----------------------------
# Имитация результатов эксперимента
# -----------------------------

np.random.seed(42)

n = experiment["sample_size_per_group"]
alpha = experiment["alpha"]

# 1 = корректное предсказание, 0 = ошибка
model_A_correct = np.random.binomial(1, 0.82, n)
model_B_correct = np.random.binomial(1, 0.87, n)

# latency в миллисекундах
model_A_latency = np.random.normal(loc=120, scale=15, size=n)
model_B_latency = np.random.normal(loc=125, scale=18, size=n)

# -----------------------------
# Проверка качества модели
# -----------------------------

correct_A = model_A_correct.sum()
incorrect_A = n - correct_A

correct_B = model_B_correct.sum()
incorrect_B = n - correct_B

table = [
    [correct_A, incorrect_A],
    [correct_B, incorrect_B]
]

odds_ratio, p_quality = fisher_exact(table, alternative="two-sided")

accuracy_A = model_A_correct.mean()
accuracy_B = model_B_correct.mean()

# -----------------------------
# Проверка технической метрики latency
# -----------------------------

t_stat, p_latency = ttest_ind(
    model_A_latency,
    model_B_latency,
    equal_var=False
)

latency_A = model_A_latency.mean()
latency_B = model_B_latency.mean()

# -----------------------------
# Вывод результатов
# -----------------------------

print("\nResults:")
print(f"Accuracy A: {accuracy_A:.3f}")
print(f"Accuracy B: {accuracy_B:.3f}")
print(f"Fisher exact test p-value: {p_quality:.4f}")

print(f"\nMean latency A: {latency_A:.2f} ms")
print(f"Mean latency B: {latency_B:.2f} ms")
print(f"t-test p-value: {p_latency:.4f}")

# -----------------------------
# Решение по A/B-тесту
# -----------------------------

quality_is_better = accuracy_B > accuracy_A and p_quality < alpha
latency_is_acceptable = latency_B <= latency_A * 1.10

print("\nDecision:")

if quality_is_better and latency_is_acceptable:
    print("Promote model B: новая модель лучше по quality, latency в допустимых пределах.")
elif p_quality >= alpha:
    print("Keep model A: статистически значимого улучшения качества не найдено.")
else:
    print("Rollback to model A: новая модель не прошла критерии качества или latency.")

A/B test parameters:
- model_A: v1.0.0 — текущая стабильная модель
- model_B: v1.1.0 — новая модель
- traffic_split: 50% / 50%
- primary_metric: accuracy
- technical_metric: latency
- alpha: 0.05
- sample_size_per_group: 300
- hypothesis_H0: Новая модель v1.1.0 не отличается от v1.0.0 по качеству.
- hypothesis_H1: Новая модель v1.1.0 показывает статистически значимое отличие по качеству.

Results:
Accuracy A: 0.823
Accuracy B: 0.847
Fisher exact test p-value: 0.5095

Mean latency A: 118.87 ms
Mean latency B: 128.11 ms
t-test p-value: 0.0000

Decision:
Keep model A: статистически значимого улучшения качества не найдено.


В A/B-тесте сравниваются две версии ML-модели: стабильная `v1.0.0` и новая `v1.1.0`.

Основная метрика - `accuracy`, то есть доля корректных предсказаний. Для сравнения correct/incorrect используется Fisher exact test. Дополнительно проверяется техническая метрика `latency` с помощью t-test.

Если новая модель статистически значимо лучше по accuracy и не ухудшает latency выше допустимого порога, её можно продвигать дальше. Если улучшение не подтверждается или технические метрики ухудшаются, остаёмся на старой модели или выполняем rollback.

## 5. Создать CI/CD-пайплайн для ML-сервиса с использованием GitHub Actions



*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*



Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

In [11]:
%%writefile deploy.yml
name: Deploy

on:
  push:
    branches: [ main ]
  workflow_dispatch:

permissions:
  contents: read
  packages: write

env:
  REGISTRY: ghcr.io

jobs:
  build-and-deploy:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Validate required secrets
        run: |
          if [ -z "${{ secrets.CLOUD_TOKEN }}" ]; then exit 1; fi

      - name: Set lowercase image name
        run: echo "IMAGE_NAME=$(echo '${{ github.repository }}' | tr '[:upper:]' '[:lower:]')" >> $GITHUB_ENV

      - name: Set up Docker Buildx
        uses: docker/setup-buildx-action@v3

      - name: Log in to GHCR
        uses: docker/login-action@v3
        with:
          registry: ${{ env.REGISTRY }}
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}

      - name: Build and push image
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: |
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ github.sha }}
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:latest

      - name: Smoke test (start + health check + predict)
        run: |
          docker run -d --name svc -p 8000:8000 \
            -e MODEL_VERSION="${{ secrets.MODEL_VERSION || 'v1.0.0' }}" \
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ github.sha }}
          for i in {1..30}; do curl -fsS http://localhost:8000/health > /dev/null 2>&1 && break || sleep 1; done
          curl -fsS http://localhost:8000/health
          curl -fsS -X POST http://localhost:8000/predict -H "Content-Type: application/json" -d '{"x":[5.1,3.5,1.4,0.2]}'
          docker stop svc

Writing deploy.yml


In [ ]:
!mkdir -p .github/workflows
!mv deploy.yml .github/workflows/deploy.yml

In [12]:
%%writefile ml_pipeline.py
import json
import os
import pickle
from pathlib import Path

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

MODEL_VERSION = os.environ.get("MODEL_VERSION", "v1.0.0")

hyperparameters = {
    "n_estimators": 100,
    "random_state": 42,
    "test_size": 0.2,
}

iris = load_iris()
X, y = iris.data, iris.target
target_names = iris.target_names.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=hyperparameters["test_size"],
    random_state=hyperparameters["random_state"],
)

model = RandomForestClassifier(
    n_estimators=hyperparameters["n_estimators"],
    random_state=hyperparameters["random_state"],
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)

with open(ARTIFACTS_DIR / "model.pkl", "wb") as f:
    pickle.dump(model, f)

with open(ARTIFACTS_DIR / "hyperparameters.json", "w") as f:
    json.dump(hyperparameters, f, indent=2)

metrics = {
    "model_version": MODEL_VERSION,
    "accuracy": accuracy,
    "n_features": X.shape[1],
    "n_classes": len(target_names),
    "target_names": target_names,
    "n_samples_train": len(X_train),
    "n_samples_test": len(X_test),
    "classification_report": report,
}
with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Model version: {MODEL_VERSION}, accuracy: {accuracy:.4f}")
print(f"Artifacts saved: model.pkl, hyperparameters.json, metrics.json")

Overwriting ml_pipeline.py


Проверяем работоспособность пайплайна

In [13]:
!python ml_pipeline.py

Model version: v1.0.0, accuracy: 1.0000
Artifacts saved: model.pkl, hyperparameters.json, metrics.json


Вам дан рабочий код пайплайна и черновик файла ci.yml. Используйте GitHub Actions и перепишите [шаг](#scrollTo=NGcDFbCFJXV_) name: Make pipeline reproducible

In [14]:
%%writefile ci.yml
name: CI

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]
  workflow_dispatch:

jobs:
  train-and-validate:
    name: Train model and validate artifacts
    runs-on: ubuntu-latest

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          cache: pip

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run ML pipeline (train + save artifacts)
        env:
          MODEL_VERSION: ${{ secrets.MODEL_VERSION || 'v1.0.0' }}
        run: python ml_pipeline.py

      - name: Make pipeline reproducible
        # Переписали этот шаг: вместо print с вопросом - реально валидируем,
        # что артефакты (модель + гиперпараметры + метрики) сохранены на диск
        run: |
          test -f artifacts/model.pkl
          test -f artifacts/hyperparameters.json
          test -f artifacts/metrics.json
          ls -la artifacts/
          cat artifacts/hyperparameters.json
          python -c "import json; m=json.load(open('artifacts/metrics.json')); print(f\"Version: {m['model_version']}, Accuracy: {m['accuracy']:.4f}\")"

      - name: Upload model artifacts
        uses: actions/upload-artifact@v4
        with:
          name: ml-model-artifacts
          path: artifacts/
          retention-days: 30

Writing ci.yml


Копируем ci.yml в правильную директорию .github/workflows

In [ ]:
!mkdir -p .github/workflows
!mv ci.yml ./.github/workflows/ci.yml

Начинаем отправку в репозиторий

In [ ]:
!git add ./.github/workflows/ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitHub Actions"
!git log

[master (root-commit) c29e765] build(ml_pipeline.py) добавлен пайплайн
 2 files changed, 35 insertions(+)
 create mode 100644 .github/workflows/ci.yml
 create mode 100644 ml_pipeline.py
commit c29e7659975b52391b4a07c33a329893f1f8426b (HEAD -> master)
Author: Your Name <you@example.com>
Date:   Wed Oct 8 19:13:08 2025 +0000

    build(ml_pipeline.py) добавлен пайплайн


После настройки workflow каждый раз при пуше в репозиторий GitHub Actions будет автоматически запускать конвейер. Пожалуйста, приложите ссылку на статус выполнения в разделе Actions **своего** репозитория на GitHub.


*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*

Ссылка на успешный workflow run в GitHub Actions:

https://github.com/TheodorWeiss/ml-model-deployment-hw7/actions

Последний зелёный запуск Deploy: собирает Docker-образ, пушит в `ghcr.io`, запускает контейнер, проверяет `/health` и `/predict` (включая negative-кейс с возвратом 400 на невалидный вход).

## 6. Итоговое оформление

В рамках задания собрал полный CI/CD-пайплайн для ML-сервиса: от обучения и сохранения артефактов (`model.pkl`, `hyperparameters.json`, `metrics.json`) до деплоя в GitHub Container Registry и автоматической проверки работоспособности через `/health` и `/predict` внутри пайплайна. Самой технически интересной частью оказалась реализация Canary через nginx с весами - простая концепция, ясный результат, всё видно в логах балансировщика и в Prometheus.

Самой сложной частью оказалась отладка bind-mount поведения Docker Desktop на Windows: попытки переключать nginx-конфигурацию «на лету» через `docker cp` или `tee` приводили к непредсказуемым сбоям из-за специфики прослойки WSL2 ↔ NTFS. В итоге переключил архитектуру на пересоздание nginx-контейнера через переменную окружения `NGINX_CONFIG` - это оказалось надёжнее и идемпотентнее.

По выбору стратегии: Canary подошёл лучше Blue-Green для этого случая, потому что в шаблонном коде ML-сервиса полностью отсутствует обработка ошибок. С Blue-Green моментальное переключение могло бы сразу обрушить весь трафик при незаметной регрессии новой версии модели; Canary же ограничивает потенциальный ущерб ~10% пользователей и даёт время заметить деградацию по метрикам.

Бонусом интегрировал Prometheus + Grafana поверх обязательного минимума - без работающего мониторинга разница между 10% canary и 100% rollout остаётся теоретической, а с ним переключения видны в реальном времени как чёткие ступеньки на графиках request rate by version. На мой взгляд, это естественное продолжение темы: стратегия деплоя без наблюдаемости бессмысленна.

